# SAADE–AUA v8.0 FINAL — Scenario-Adaptive **Evaluation** of Audio Deepfake Detectors

**Завдання:** сценарно-адаптивне **тестування готових, зафіксованих моделей**, НЕ навчання. Методологічний орієнтир: [Lipianina-Honcharenko et al., MAKE 2026, DOI 10.3390/make8060161](https://doi.org/10.3390/make8060161): контрольовані сценарії → фіксований інференс → незалежне оцінювання → профілі ризиків і багатокритеріальне порівняння.

- Джерело: уже створені у v6.2 `V6_LOCKED_MATCHED_SPLITS.csv` і WAV; жодних нових TTS і жодних змін AUA.
- Сценарії: чистий сигнал + 5 рівнів шуму, MP3, штучної реверберації та lowpass = **21**.
- Детектори: pretrained AASIST, AASIST-L, Wav2Vec2 Stafford, **лише `eval()` і inference**, не змінюємо ваги.
- Результати: BA, ROC-AUC, EER, F1, ASRI-Audio (адаптація ARI), парні CI, ранжування, MCDM, графіки, ZIP.
- **Публікаційне обмеження:** validation містить один синтетичний голос (`tetiana`), тому CI умовні для цього голосу; цей експеримент **не підтверджує** узагальнення на нові TTS-рушії або довільні українські голоси.
- Hidden test не відкривається при звичайному `Run All`. Його запуск має окрему процедуру заморожування з подвійною згодою.

**Послідовність:** режим `VALIDATION` → перевірка результатів → окремий `FREEZE` → лише після рішення дослідника `HIDDEN_ONCE`.

## 0. Конфігурація — змінюйте лише цю комірку

In [ ]:
from pathlib import Path
PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'   # READ ONLY
RESULTS_ROOT = PROJECT / 'results'

# SMOKE = 12 пар; VALIDATION = усі validation-пари;
# FREEZE = фіксація результатів повної validation;
# HIDDEN_ONCE = одноразовий незалежний тест після FREEZE.
RUN_PHASE = 'VALIDATION'  # 'SMOKE', 'VALIDATION', 'FREEZE', 'HIDDEN_ONCE'
SMOKE_PAIRS = 12
MODEL_ENABLED = {'AASIST_ONNX':True, 'AASIST_L_ONNX':True, 'WAV2VEC2_STAFFORD':True}
CALIBRATION_PAIRS = 48    # відбір лише з train; без оновлення ваг
BOOTSTRAP_REPS = 1000     # для SMOKE автоматично обмежено 200
VERIFY_SHA256 = True
SEED = 20261115
INSTALL_MISSING = False
SHOW_TRAIN_AUDIO_EXAMPLE = True
CACHE_MP3_TRANSFORMS = True   # повторне використання однакових MP3-версій між моделями

# НІКОЛИ НЕ вмикайте для звичайного повного запуску.
FREEZE_CONFIRMATION = 'NO'    # для FREEZE: 'I_APPROVE_THE_VALIDATION_PROTOCOL'
HIDDEN_CONFIRMATION = 'NO'    # для HIDDEN_ONCE: 'I_AUTHORIZE_ONE_FINAL_HIDDEN_TEST'

## 1. Імпорти, залежності, робочі каталоги

In [ ]:
import os, io, sys, gc, json, math, time, zipfile, hashlib, shutil, subprocess
import tempfile, urllib.request, warnings, itertools
from fractions import Fraction
import numpy as np
import pandas as pd
from IPython.display import display, Audio

assert RUN_PHASE in {'SMOKE','VALIDATION','FREEZE','HIDDEN_ONCE'}, f'Невідомий RUN_PHASE: {RUN_PHASE}'
assert AUA.exists(), f'Не знайдено AUA: {AUA}'
assert RESULTS_ROOT.exists() or PROJECT.exists(), 'Не знайдено робочу папку проекту'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
V6_CANDIDATES = [RESULTS_ROOT/'SAADE_AUA_2026_v6_2_MATCHED_TTS',
                 RESULTS_ROOT/'SAADE_AUA_2026_v6_MATCHED_TTS']
V6_DIR = next((d for d in V6_CANDIDATES if (d/'V6_LOCKED_MATCHED_SPLITS.csv').is_file()), None)
if V6_DIR is None:
    raise FileNotFoundError('Немає V6_LOCKED_MATCHED_SPLITS.csv у v6.2 або v6. Спершу завершити реальну генерацію TTS.')
BASE = RESULTS_ROOT/'SAADE_AUA_2026_v8_FINAL_EVALUATION'
RUN_TAG = f'SMOKE_{SMOKE_PAIRS}' if RUN_PHASE=='SMOKE' else 'FULL_VALIDATION'
OUT = BASE / RUN_TAG
OUT.mkdir(parents=True,exist_ok=True)
MODEL_CACHE = BASE/'pretrained_model_cache'
MODEL_CACHE.mkdir(parents=True,exist_ok=True)

if INSTALL_MISSING:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy','pandas','scipy',
        'soundfile','scikit-learn','matplotlib','torch','transformers>=4.40,<5',
        'onnxruntime','huggingface_hub','statsmodels','safetensors'])
import soundfile as sf
from scipy import signal, stats
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['figure.figsize'] = (10,5)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path,obj):
    path=Path(path); tmp=path.with_name(path.name+'.part')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    os.replace(tmp,path)

def normhash(obj):
    return hashlib.sha256(json.dumps(obj,ensure_ascii=False,sort_keys=True,default=str).encode()).hexdigest()

PROTOCOL_ID = 'saade-v8-20261115-4s-16k-same-noise-per-pair-active-only-21-scenarios'
MAX_PAIRS = SMOKE_PAIRS if RUN_PHASE=='SMOKE' else None
if RUN_PHASE in ('FREEZE','HIDDEN_ONCE') and MAX_PAIRS is not None:
    raise RuntimeError('Заборонено заморожувати або відкривати hidden після SMOKE')
print('Фаза:',RUN_PHASE,'| результати:',OUT,'| аудіодані:',V6_DIR)
print('Залежності готові. FFMPEG:',shutil.which('ffmpeg'))

## 2. Маніфест пар, аудиторські перевірки та незалежність split

In [ ]:
MANIFEST_PATH = V6_DIR/'V6_LOCKED_MATCHED_SPLITS.csv'
raw = pd.read_csv(MANIFEST_PATH,low_memory=False)
required={'path','label','pair_id','split','tts_voice_id','group_id','parent_file_id'}
assert required.issubset(raw.columns),f'Відсутні стовпці: {required-set(raw.columns)}'
raw['label']=pd.to_numeric(raw['label'],errors='raise').astype(int)
raw['pair_id']=raw.pair_id.astype(str)
assert raw.label.isin([0,1]).all() and raw.pair_id.notna().all()
assert set(raw.split.unique()).issubset({'train','adaptation','validation','hidden_test'}), 'Невідомий split'
assert raw.groupby('pair_id').split.nunique().max()==1, 'PAIR LEAKAGE між split'
# Фізичні TTS voice groups, наскільки їх визначено у v6, не можуть повторюватися між split.
fake_groups=raw.loc[raw.label.eq(1)].dropna(subset=['group_id'])
voice_overlaps=fake_groups.groupby('group_id').split.nunique()
if (voice_overlaps>1).any():
    raise RuntimeError(f'FAKE VOICE GROUP LEAKAGE: {int((voice_overlaps>1).sum())} груп між split')
for col in ['group_id','parent_file_id','text_key','input_text_id','working_sha256']:
    if col not in raw.columns:continue
    keys=raw[col].dropna().astype(str)
    t=raw.loc[keys.index].copy()
    if col=='group_id':t=t.loc[t.label.eq(0)] if col=='group_id' else t
    cross=t.groupby(col).split.nunique()
    if (cross>1).any():raise RuntimeError(f'LEAKAGE {col}: {int((cross>1).sum())} ID у різних split')

pub=raw.loc[raw.split.ne('hidden_test')].copy()  # жодне hidden audio не відкриваємо
manifest=pub.loc[pub.split.eq('validation')].copy()
assert not manifest.empty,'Відсутня validation'
valid_pairs=manifest.groupby(['pair_id','label']).size().unstack(fill_value=0)
assert {0,1}.issubset(valid_pairs.columns) and valid_pairs[[0,1]].eq(1).all().all(), 'Порушені REAL–TTS пари'
if MAX_PAIRS is not None:
    chosen=np.random.default_rng(SEED).choice(sorted(manifest.pair_id.unique()),size=min(MAX_PAIRS,len(valid_pairs)),replace=False)
    manifest=manifest.loc[manifest.pair_id.isin(chosen)].copy()
assert len(manifest)==2*manifest.pair_id.nunique()
assert manifest.groupby('pair_id').label.nunique().eq(2).all()

def check_file_hashes(rows,stage):
    missing=[str(p) for p in rows.path.unique() if not Path(p).is_file()]
    if missing:raise FileNotFoundError(f'{stage}: немає {len(missing)} WAV, зокрема: {missing[:3]}')
    receipt_file=V6_DIR/'V6_TTS_RECEIPTS.csv'
    if not receipt_file.is_file():raise FileNotFoundError('Немає V6_TTS_RECEIPTS.csv')
    rec=pd.read_csv(receipt_file)
    need_receipts={'pair_id','generated_path','sha256'}
    assert need_receipts.issubset(rec.columns),f'Некоректний receipts {need_receipts-set(rec.columns)}'
    receipt_map={(str(r.pair_id),str(r.generated_path)):str(r.sha256).lower()
                 for r in rec.itertuples(index=False)}
    records=[]
    for r in rows.itertuples(index=False):
        expected=(receipt_map.get((str(r.pair_id),str(r.path))) if int(r.label)==1
                  else getattr(r,'working_sha256',None))
        if expected is None or pd.isna(expected) or len(str(expected))!=64:
            raise RuntimeError(f'{stage}: відсутня підтверджена SHA-256 для {r.pair_id}, class={r.label}')
        actual=sha256_file(r.path) if VERIFY_SHA256 else None
        if VERIFY_SHA256 and actual!=str(expected).lower():
            raise RuntimeError(f'{stage}: SHA-256 не збігається у pair={r.pair_id} class={r.label}')
        records.append(dict(pair_id=str(r.pair_id),label=int(r.label),sha256_verified=bool(VERIFY_SHA256)))
    return pd.DataFrame(records)

# Публічні WAV перевіряємо, hidden — лише на етапі HIDDEN_ONCE.
check_file_hashes(manifest,'validation').to_csv(OUT/'01_VALIDATION_HASH_AUDIT.csv',index=False)
voice_counts=raw.loc[raw.label.eq(1)].groupby(['split','tts_voice_id']).pair_id.nunique().reset_index(name='pairs')
voice_counts.to_csv(OUT/'01_VOICE_SPLITS.csv',index=False)
voice_count_val=int(voice_counts[voice_counts.split.eq('validation')].tts_voice_id.nunique())
readiness={'validation_pairs_total':int(raw.loc[raw.split.eq('validation')].pair_id.nunique()),
           'selected_validation_pairs':int(manifest.pair_id.nunique()),
           'independent_fake_voice_ids_validation':voice_count_val,
           'hidden_test_opened':False,'pretrained_only':True,
           'scope':'within one known TTS voice on this controlled Ukrainian matched-text corpus; NOT cross-generator generalization'}
write_json_atomic(OUT/'01_READINESS.json',readiness)
print('Validation:',len(manifest)//2,'пар. Синтетичних голосових груп у validation:',voice_count_val)
display(voice_counts)

# Ніяких повних шляхів/текстів у звітному ZIP, лише локально на сервері.
manifest.to_csv(OUT/'01_PRIVATE_VALIDATION_MANIFEST.csv',index=False)

## 3. Однакові аудіовікна та відтворювані сценарії

In [ ]:
assert shutil.which('ffmpeg'),'FFmpeg обов’язковий для справжнього MP3 encode→decode'
SR=16000; WINDOW_SEC=4.0; WINDOW_N=int(SR*WINDOW_SEC)
SCENARIOS=[{'scenario':'clean','family':'clean','level':0,'parameter':None}]
for level,snr in enumerate([30,25,20,15,10],1):
    SCENARIOS.append({'scenario':f'noise_snr{snr}','family':'noise','level':level,'parameter':snr})
for level,kbps in enumerate([128,96,64,48,32],1):
    SCENARIOS.append({'scenario':f'mp3_{kbps}kbps','family':'mp3','level':level,'parameter':kbps})
for level,rt in enumerate([.15,.30,.45,.65,.85],1):
    SCENARIOS.append({'scenario':f'reverb_rt{rt:.2f}','family':'reverb','level':level,'parameter':rt})
for level,hz in enumerate([7500,6000,4500,3000,2000],1):
    SCENARIOS.append({'scenario':f'lowpass_{hz}Hz','family':'lowpass','level':level,'parameter':hz})
SC=pd.DataFrame(SCENARIOS)
assert len(SC)==21 and SC.scenario.is_unique
SC.to_csv(OUT/'02_SCENARIO_PROTOCOL.csv',index=False)
SCENARIO_HASH=normhash(SCENARIOS)

def stable_seed(*parts):
    return int.from_bytes(hashlib.sha256('||'.join(map(str,parts)).encode()).digest()[:8],'big')%(2**32)

# Тривалість вирівнюється всередині пари до обрізання 4 с.
# Перевіряємо train, що використовується лише для калібрування, і validation.
public_used=pub.loc[pub.split.eq('train') | pub.pair_id.isin(manifest.pair_id)].copy()
pair_active={}
for pid,g in public_used.groupby('pair_id'):
    if set(g.label)!={0,1}:continue
    durations=[]
    for r in g.itertuples(index=False):
        info=sf.info(str(r.path))
        start=float(getattr(r,'segment_start_sec',0) or 0)
        dur=float(info.duration)-start
        if not np.isfinite(dur) or dur<0.5:raise ValueError(f'Дуже короткий WAV: {pid}, {dur}')
        durations.append(dur)
    pair_active[str(pid)]=int(max(1,min(WINDOW_N,math.floor(min(durations)*SR))))
assert set(manifest.pair_id).issubset(pair_active),'Не вдалося вирівняти всі validation-пари'
pd.DataFrame([{'pair_id':k,'effective_duration_sec':v/SR} for k,v in pair_active.items()]).to_csv(
    OUT/'02_PAIR_DURATION_POLICY.csv',index=False)

def read_wave(path,start_sec=0,pair_id=None):
    with sf.SoundFile(str(path)) as f:
        sr=f.samplerate
        f.seek(max(0,int(round(float(start_sec)*sr))))
        x=f.read(frames=int(math.ceil(WINDOW_SEC*sr)),dtype='float32',always_2d=True).mean(axis=1)
    if sr!=SR:
        fr=Fraction(SR,sr)
        x=signal.resample_poly(x,fr.numerator,fr.denominator)
    x=np.nan_to_num(np.asarray(x,dtype=np.float32),nan=0.,posinf=0.,neginf=0.)
    if pair_id is not None:x=x[:pair_active[str(pair_id)]]
    if len(x)==0:raise ValueError('Порожній WAV: '+str(path))
    return np.pad(x[:WINDOW_N],(0,max(0,WINDOW_N-len(x))))[:WINDOW_N].astype(np.float32)

def mp3_roundtrip(active,kbps):
    with tempfile.TemporaryDirectory(prefix='saade_mp3_') as tmp:
        wd=Path(tmp); src=wd/'original.wav'; mp3=wd/'compressed.mp3'; dec=wd/'decoded.wav'
        sf.write(src,active,SR,subtype='PCM_16')
        args=['ffmpeg','-nostdin','-y','-hide_banner','-loglevel','error']
        subprocess.run(args+['-i',str(src),'-c:a','libmp3lame','-b:a',f'{kbps}k',
                   '-ar',str(SR),'-ac','1',str(mp3)],check=True,timeout=60,stderr=subprocess.PIPE)
        subprocess.run(args+['-i',str(mp3),'-ar',str(SR),'-ac','1',str(dec)],
                   check=True,timeout=60,stderr=subprocess.PIPE)
        y,sr=sf.read(dec,dtype='float32',always_2d=True)
        assert sr==SR
        y=y.mean(axis=1)
    return np.pad(y[:len(active)],(0,max(0,len(active)-len(y))))[:len(active)]

def apply_scenario(wave,sc,pair_id,active_n=None):
    """Perturb ONLY active audio, keep deterministic equal padding per REAL/FAKE pair."""
    n=WINDOW_N if active_n is None else int(active_n)
    if n<1 or n>WINDOW_N:raise ValueError('Bad active_n')
    active=wave[:n].copy()
    family=sc['family']; val=sc['parameter']
    if family=='clean':out=active
    elif family=='noise':
        # ОДНАКОВИЙ шумовий шаблон для REAL та FAKE однієї пари, SNR залежить від RMS кожного сигналу.
        rng=np.random.default_rng(stable_seed(PROTOCOL_ID,pair_id,sc['scenario']))
        z=rng.standard_normal(n).astype(np.float32)
        z/=np.sqrt(np.mean(z.astype(np.float64)**2)+1e-12)
        rms=np.sqrt(np.mean(active.astype(np.float64)**2)+1e-12)
        out=np.clip(active+z*(rms/(10**(float(val)/20))),-1,1)
    elif family=='mp3':out=mp3_roundtrip(active,int(val))
    elif family=='reverb':
        # Синтетичний IR, НЕ емпірично виміряний RT60 кімнати.
        rt=float(val);m=int(min(SR*1.5,SR*rt*1.2))
        t=np.arange(m)/SR;env=np.exp(-6.9078*t/rt)
        rng=np.random.default_rng(stable_seed(PROTOCOL_ID,'IR',sc['scenario']))
        h=rng.normal(size=m).astype(np.float32)*env
        h[0]+=4.0
        h/=np.sqrt(np.sum(h.astype(np.float64)**2)+1e-12)
        out=np.clip(signal.fftconvolve(active,h,mode='full')[:n],-1,1)
    elif family=='lowpass':
        sos=signal.butter(6,float(val),fs=SR,output='sos',btype='lowpass')
        out=signal.sosfilt(sos,active)
    else:raise KeyError(family)
    out=np.asarray(out,dtype=np.float32)
    assert out.shape==(n,) and np.isfinite(out).all()
    return np.pad(out,(0,WINDOW_N-n)).astype(np.float32)

probe=(0.12*np.sin(2*np.pi*440*np.arange(WINDOW_N)/SR)).astype(np.float32)
for sc in SCENARIOS:
    y=apply_scenario(probe,sc,'test_pair',int(3*SR))
    assert y.shape==(WINDOW_N,) and np.isfinite(y).all() and not y[int(3*SR):].any()
assert np.array_equal(apply_scenario(probe,SCENARIOS[1],'p',SR),apply_scenario(probe,SCENARIOS[1],'p',SR))
def transform_cached(wave,sc,row,cache_root):
    """Cache deterministic MP3 only; cache fingerprint includes source SHA and scenario.
    This cache is never packaged into the review archive.
    """
    active_n=pair_active[str(row.pair_id)]
    if not CACHE_MP3_TRANSFORMS or sc['family']!='mp3':
        return apply_scenario(wave,sc,str(row.pair_id),active_n)
    key=normhash({'protocol':PROTOCOL_ID,'source_sha':str(row.working_sha256),
                  'pair':str(row.pair_id),'label':int(row.label),
                  'scenario':sc['scenario'],'active_n':active_n})
    dest=Path(cache_root)/(key+'.npy')
    if dest.is_file():
        arr=np.load(dest,allow_pickle=False)
        if arr.shape!=(WINDOW_N,) or arr.dtype!=np.float32 or not np.isfinite(arr).all():
            raise RuntimeError('Пошкоджено кеш MP3: '+str(dest))
        return arr
    out=apply_scenario(wave,sc,str(row.pair_id),active_n)
    dest.parent.mkdir(parents=True,exist_ok=True)
    tmp=dest.with_name(dest.name+'.part.npy')
    np.save(tmp,out,allow_pickle=False)
    os.replace(tmp,dest)
    return out

print('Сценарний self-test: 21/21; усе детерміновано; спільна довжина й паддинг')

## 4. Готові pretrained-моделі: верифіковані checkpoint, фіксована орієнтація score

In [ ]:
MODEL_SOURCES={
 'AASIST_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST',
                'file':'aasist.onnx','source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST'},
 'AASIST_L_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST-L',
                  'file':'aasist-l.onnx','source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L'},
 'WAV2VEC2_STAFFORD':{'kind':'hf','repo':'garystafford/wav2vec2-deepfake-voice-detector',
                     'expected_fake_index':1,
                     'source':'https://huggingface.co/garystafford/wav2vec2-deepfake-voice-detector'}
}
SELECTED_MODELS=[m for m,v in MODEL_ENABLED.items() if v]
if not SELECTED_MODELS:raise ValueError('Жодної моделі')
if not set(SELECTED_MODELS).issubset(MODEL_SOURCES):raise ValueError('Модель не має перевіреного адаптера')

def download_identity(repo,filename,dest,min_bytes=50000):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.is_file() and dest.stat().st_size>=min_bytes:return dest
    url=f'https://huggingface.co/{repo}/resolve/main/{filename}'
    req=urllib.request.Request(url,headers={'User-Agent':'SAADE-v8-research/1.0','Accept-Encoding':'identity'})
    part=dest.with_name(dest.name+'.part')
    try:
        with urllib.request.urlopen(req,timeout=180) as response, open(part,'wb') as f:
            while True:
                b=response.read(2*1024*1024)
                if not b:break
                f.write(b)
        if part.stat().st_size<min_bytes or part.open('rb').read(32).lstrip().lower().startswith(b'<html'):
            raise RuntimeError('Файл моделі не завантажено повністю: '+url)
        os.replace(part,dest)
    finally:part.unlink(missing_ok=True)
    return dest

def fake_index_from_labels(id2label,expected):
    labels={int(k):str(v).strip().lower() for k,v in id2label.items()}
    assert set(labels)=={0,1},'Не 2 класи: '+repr(labels)
    explicit=[k for k,v in labels.items() if 'fake' in v or 'spoof' in v or 'synthetic' in v]
    if len(explicit)==1 and explicit[0]!=expected:
        raise RuntimeError('Класи моделі не збігаються з документацією: '+str(labels))
    if len(explicit)==0 and set(labels.values())!={'label_0','label_1'}:
        raise RuntimeError('Неможливо підтвердити мітки FAKE/REAL: '+str(labels))
    if expected not in labels:raise ValueError('Неправильний FAKE index')
    return expected,labels

class OnnxAASIST:
    def __init__(self,name,cfg):
        import onnxruntime as ort
        self.name=name;self.source=cfg['source']
        fp=download_identity(cfg['repo'],cfg['file'],MODEL_CACHE/cfg['file'])
        self.weights_sha256=sha256_file(fp)
        self.model_revision=self.weights_sha256
        self.session=ort.InferenceSession(str(fp),providers=['CPUExecutionProvider'])
        ins=self.session.get_inputs()
        assert len(ins)==1,'Непідтримуваний ONNX input'
        self.input_name=ins[0].name
        self.input_shape=ins[0].shape
        self.score_kind='negative_bonafide_logit (higher is fake, not probability)'
        self.labels={0:'bonafide',1:'spoof'}
    def score_fake(self,w):
        x=np.asarray(w[:64600],dtype=np.float32)
        if len(x)<64600:x=np.pad(x,(0,64600-len(x)))
        sh=self.input_shape
        if len(sh)==2:inp=x[None,:]
        elif len(sh)==3 and sh[1] in (1,'1',None):inp=x[None,None,:]
        else:raise RuntimeError('Невідомий shape ONNX: '+str(sh))
        outs=self.session.run(None,{self.input_name:inp})
        vals=np.asarray(outs[0],dtype=float).reshape(-1)
        if vals.size==1:bonafide=vals[0]
        elif vals.size==2:bonafide=vals[1]
        else:raise RuntimeError('Непередбачений вихід AASIST: '+str(vals.shape))
        return float(-bonafide)
    def close(self):del self.session


def locate_hf_weights(repo,revision=None,fallback=None):
    """Find exact cached weights to fingerprint. No unknown revision is accepted."""
    candidates=[]
    try:
        from huggingface_hub import constants
        cache=Path(constants.HF_HUB_CACHE)
        root=cache/('models--'+repo.replace('/','--'))/'snapshots'
        if revision and revision!='unknown':
            q=root/revision/'model.safetensors'
            if q.exists():candidates.append(q)
        else:
            candidates.extend(root.glob('*/model.safetensors'))
    except Exception:pass
    if fallback and Path(fallback).exists():candidates.insert(0,Path(fallback))
    # Multiple ambiguous snapshots MUST NOT be chosen using mtime alone.
    hashes={sha256_file(p):p for p in candidates if p.is_file() and p.stat().st_size>50_000}
    if len(hashes)!=1:
        raise RuntimeError(f'Неможливо однозначно зафіксувати SHA-256 checkpoint {repo}. '
                           f'Знайдено варіантів: {len(hashes)}. Приберіть зайві версії кешу або '
                           'завантажте модель в окремий локальний каталог.')
    return next(iter(hashes)), str(next(iter(hashes.values())))

class HFWav2Vec2:
    def __init__(self,name,cfg):
        import torch
        from transformers import AutoFeatureExtractor,AutoModelForAudioClassification
        self.name=name;self.source=cfg['source'];self.torch=torch
        self.device='cuda' if torch.cuda.is_available() else 'cpu'
        self._cache_weights=None
        try:
            self.fe=AutoFeatureExtractor.from_pretrained(cfg['repo'],trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(cfg['repo'],trust_remote_code=False)
        except Exception as exc:
            print('HF download problem; HTTPS identity fallback:',str(exc)[:180])
            folder=MODEL_CACHE/name
            for f in ['config.json','preprocessor_config.json','model.safetensors']:
                download_identity(cfg['repo'],f,folder/f,50000 if f.endswith('safetensors') else 100)
            self._cache_weights=folder/'model.safetensors'
            self.fe=AutoFeatureExtractor.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
        self.fake_idx,self.labels=fake_index_from_labels(self.model.config.id2label,cfg['expected_fake_index'])
        commit=getattr(self.model.config,'_commit_hash',None)
        self.weights_sha256,self.weights_file=locate_hf_weights(cfg['repo'],commit,self._cache_weights)
        self.model_revision=self.weights_sha256
        self.model=self.model.to(self.device).eval()
        self.model.requires_grad_(False)
        self.score_kind='softmax_fake_probability'
    def score_fake(self,w):
        batch=self.fe(w,sampling_rate=SR,return_tensors='pt',padding=True)
        batch={k:v.to(self.device) for k,v in batch.items()}
        with self.torch.inference_mode():
            probs=self.torch.softmax(self.model(**batch).logits,dim=-1)
        return float(probs[0,self.fake_idx].item())
    def close(self):
        del self.model
        if self.device=='cuda':self.torch.cuda.empty_cache()

def load_detector(name):
    cfg=MODEL_SOURCES[name]
    return OnnxAASIST(name,cfg) if cfg['kind']=='onnx' else HFWav2Vec2(name,cfg)

print('Обрано готові моделі:',SELECTED_MODELS)
print('Інференс без навчання: ONNX CPU / Torch eval + inference_mode.')

## 5. Фіксація порогів на train (не навчання моделі)

In [ ]:
def calibration_rows():
    t=pub.loc[pub.split.eq('train')].copy()
    ids=sorted(t.pair_id.unique())
    if not ids:raise RuntimeError('Немає train для калібрування порогів')
    chosen=np.random.default_rng(SEED+1).choice(ids,size=min(CALIBRATION_PAIRS,len(ids)),replace=False)
    t=t.loc[t.pair_id.isin(chosen)].copy()
    counts=t.groupby(['pair_id','label']).size().unstack(fill_value=0)
    assert set(counts.columns)=={0,1} and counts[[0,1]].eq(1).all().all()
    assert not set(t.pair_id).intersection(set(manifest.pair_id))
    missing=[p for p in t.path if not Path(p).exists()]
    if missing:raise FileNotFoundError('Відсутнє калібрувальне WAV: '+str(missing[0]))
    return t

calibration=calibration_rows()
check_file_hashes(calibration,'train').to_csv(OUT/'01_TRAIN_HASH_AUDIT.csv',index=False)
CALIB_HASH=normhash(sorted(calibration.pair_id.unique()))

def choose_threshold(y,score):
    y=np.asarray(y,dtype=int);score=np.asarray(score,dtype=float)
    if set(np.unique(y))!={0,1} or not np.isfinite(score).all():raise ValueError('Invalid train scores')
    vals=np.unique(score)
    if len(vals)==1:return float(vals[0]),0.5
    grid=np.r_[np.nextafter(vals[0],-np.inf),(vals[1:]+vals[:-1])/2,
               np.nextafter(vals[-1],np.inf)]
    bas=np.array([balanced_accuracy_score(y,score>=p) for p in grid])
    ties=np.flatnonzero(np.isclose(bas,bas.max()))
    best=grid[ties[len(ties)//2]]
    return float(best),float(bas.max())

RUN_CONFIG={'protocol_id':PROTOCOL_ID,'run_tag':RUN_TAG,'seed':SEED,
           'manifest_sha256':sha256_file(MANIFEST_PATH),
           'validation_pair_ids_sha256':normhash(sorted(manifest.pair_id.unique())),
           'calibration_pair_ids_sha256':CALIB_HASH,
           'models':SELECTED_MODELS,'scenario_hash':SCENARIO_HASH,'sr':SR,
           'window_sec':WINDOW_SEC,'calibration_pairs':CALIBRATION_PAIRS,
           'noise_policy':'shared pair noise waveform, relative SNR, active audio only',
           'padding_policy':'pairwise min duration, zero-pad to four seconds',
           'mp3_cache_enabled':CACHE_MP3_TRANSFORMS}
CONFIG_PATH=OUT/'00_RUN_CONFIG.json'
if CONFIG_PATH.exists():
    prev=json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    if prev!=RUN_CONFIG:
        raise RuntimeError('Конфігурація відрізняється від попереднього запуску в цьому OUT. '
                           'Не змішуйте кеші: змініть ім’я результатів або відновіть стару конфігурацію.')
else:write_json_atomic(CONFIG_PATH,RUN_CONFIG)

PRED_PATH=OUT/'05_VALIDATION_PREDICTIONS.csv'
CAL_PATH=OUT/'04_CALIBRATION.json'
PROV_PATH=OUT/'03_MODEL_PROVENANCE.json'
if PRED_PATH.exists():
    predictions=pd.read_csv(PRED_PATH,low_memory=False)
else:predictions=pd.DataFrame()
cal_meta=json.loads(CAL_PATH.read_text()) if CAL_PATH.exists() else {}
model_provenance=json.loads(PROV_PATH.read_text()) if PROV_PATH.exists() else {}
print('Калібрувальні REAL–TTS пари:',calibration.pair_id.nunique())

## 6. Фіксований інференс, відновлюваність, сувора перевірка checksum

In [ ]:
def save_prediction_rows(items):
    global predictions
    if not items:return
    part=pd.DataFrame(items)
    if predictions.empty:predictions=part
    else:predictions=pd.concat([predictions,part],ignore_index=True)
    key=['protocol_id','model','model_revision','pair_id','label','scenario']
    predictions=predictions.drop_duplicates(key,keep='last')
    tmp=PRED_PATH.with_suffix('.part')
    predictions.to_csv(tmp,index=False)
    os.replace(tmp,PRED_PATH)

def eval_model_validation(name):
    print('Pretrained model:',name)
    detector=load_detector(name)
    try:
        digest=detector.weights_sha256
        if not digest or len(digest)!=64:raise RuntimeError('Немає SHA-256 реального checkpoint')
        if name in model_provenance and model_provenance[name]['weights_sha256']!=digest:
            raise RuntimeError('Checkpoint змінено відносно старого кешу! Не можна об’єднувати прогнози.')
        model_provenance[name]={'source':detector.source,'weights_sha256':digest,
                                'model_revision':detector.model_revision,'score_kind':detector.score_kind,
                                'id2label':detector.labels,'training_performed':False}
        write_json_atomic(PROV_PATH,model_provenance)
        scores=[]
        for r in calibration.itertuples(index=False):
            w=read_wave(r.path,getattr(r,'segment_start_sec',0),r.pair_id)
            scores.append(detector.score_fake(w))
        thr,train_ba=choose_threshold(calibration.label.values,scores)
        train_auc=float(roc_auc_score(calibration.label.values,scores))
        if name in cal_meta:
            before=cal_meta[name]
            if not np.isclose(float(before['threshold']),thr,atol=1e-7,rtol=1e-7) or before['weights_sha256']!=digest:
                raise RuntimeError(f'{name}: змінилась калібрувальна конфігурація або модель')
        cal_meta[name]={'threshold':thr,'train_BA':train_ba,'train_ROC_AUC':train_auc,
                        'train_pairs':int(calibration.pair_id.nunique()),
                        'weights_sha256':digest,'split':'train','no_model_training':True}
        write_json_atomic(CAL_PATH,cal_meta)
        if train_auc<=0.5:
            print('УВАГА: train ROC-AUC <= 0.5. Перевірити орієнтацію класів, не інвертувати автоматично.')
        print(f'Поріг train={thr:.6f}; BA={train_ba:.3f}; AUC={train_auc:.3f}')
        if not predictions.empty:
            other=predictions.loc[predictions.model.eq(name)]
            if len(other) and (other.model_revision.astype(str)!=digest).any():
                raise RuntimeError('Існують кешовані прогнози з іншими вагами моделі')
            if len(other) and not np.allclose(other.threshold.astype(float),thr,atol=1e-7,rtol=1e-7):
                raise RuntimeError('Існують кешовані прогнози з іншим порогом')
        seen=set()
        if not predictions.empty:
            old=predictions.loc[predictions.model.eq(name)]
            seen=set(zip(old.pair_id.astype(str),old.label.astype(int),old.scenario.astype(str)))
        pending=[];count=0;t0=time.time()
        for r in manifest.itertuples(index=False):
            todo=[s for s in SCENARIOS if (str(r.pair_id),int(r.label),s['scenario']) not in seen]
            if not todo:continue
            w=read_wave(r.path,getattr(r,'segment_start_sec',0),r.pair_id)
            n=pair_active[str(r.pair_id)]
            for s in todo:
                z=transform_cached(w,s,r,OUT/'_mp3_cache')
                p=detector.score_fake(z)
                if not np.isfinite(p):raise RuntimeError(f'NaN/Inf score: {name} {r.pair_id} {s["scenario"]}')
                pending.append({'protocol_id':PROTOCOL_ID,'model':name,'model_revision':digest,
                                'pair_id':str(r.pair_id),'label':int(r.label),'scenario':s['scenario'],
                                'family':s['family'],'level':int(s['level']),
                                'score_fake':float(p),'threshold':thr,'pred_label':int(p>=thr),
                                'tts_voice_id':str(r.tts_voice_id)})
                count+=1
            if len(pending)>=84:save_prediction_rows(pending);pending=[]
        save_prediction_rows(pending)
        print(f'Обчислено {count} нових прогнозів; час {time.time()-t0:.1f} с')
    finally:
        detector.close();del detector;gc.collect()

# У FREEZE або HIDDEN_ONCE не оцінюємо validation наново, лише підтверджуємо готовий кеш.
if RUN_PHASE in {'SMOKE','VALIDATION'}:
    for modelname in SELECTED_MODELS:eval_model_validation(modelname)
else:print('Фаза',RUN_PHASE,'— validation-прогнози не перераховуються')
if predictions.empty:raise RuntimeError('Відсутні validation-прогнози. Спочатку виконайте RUN_PHASE=VALIDATION')
expected=int(2*manifest.pair_id.nunique()*len(SCENARIOS))
for name in SELECTED_MODELS:
    part=predictions.loc[predictions.model.eq(name)]
    if len(part)!=expected:raise RuntimeError(f'{name}: {len(part)}/{expected} прогнозів; НЕ завершено')
    if part.protocol_id.nunique()!=1 or str(part.protocol_id.iloc[0])!=PROTOCOL_ID:
        raise RuntimeError('Прогнози іншого протоколу')
    assert len(part.drop_duplicates(['pair_id','label','scenario']))==expected,'Дублікати прогнозів'
    assert set(part.pair_id.astype(str))==set(manifest.pair_id.astype(str)),'Кеш не від цієї validation'
    assert part.threshold.nunique()==1,'Пороги змішано'
    if name not in model_provenance or name not in cal_meta:raise RuntimeError('Немає provenance або calibration')
    assert (part.model_revision.astype(str)==model_provenance[name]['weights_sha256']).all()
    assert np.isclose(part.threshold.iloc[0],cal_meta[name]['threshold'])
assert len(predictions)==expected*len(SELECTED_MODELS),'Зайві/чужі записи в кеші'
print('ПЕРЕВІРЕНО:',len(predictions),'прогнозів, 21 сценарій, жодного тренування')

## 7. Показники, індекс ASRI-Audio та сценарні вразливості

In [ ]:
def eer_score(y,score):
    y=np.asarray(y,dtype=int);score=np.asarray(score,dtype=float)
    if len(set(y))<2:return np.nan
    fpr,tpr,_=roc_curve(y,score)
    fnr=1-tpr
    diff=fpr-fnr
    cross=np.flatnonzero(diff[:-1]*diff[1:]<=0)
    if len(cross):
        i=int(cross[0]);delta=diff[i+1]-diff[i]
        lam=0.0 if abs(delta)<1e-15 else float(-diff[i]/delta)
        return float(fpr[i]+lam*(fpr[i+1]-fpr[i]))
    k=int(np.argmin(abs(diff)))
    return float((fpr[k]+fnr[k])/2)

def evaluate_metrics(frame):
    y=frame.label.to_numpy(int);pred=frame.pred_label.to_numpy(int)
    score=frame.score_fake.to_numpy(float)
    assert set(y)=={0,1},'Сценарій без обох класів'
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'pairs':int(frame.pair_id.nunique()),'n_real':int((y==0).sum()),
            'n_fake':int((y==1).sum()),'BA':float(balanced_accuracy_score(y,pred)),
            'ROC_AUC':float(roc_auc_score(y,score)),'EER':float(eer_score(y,score)),
            'F1_fake':float(f1_score(y,pred,zero_division=0)),
            'TPR_fake':float(tp/(tp+fn)),'TNR_real':float(tn/(tn+fp)),
            'TP':int(tp),'TN':int(tn),'FP':int(fp),'FN':int(fn)}

def analyze_predictions(data,folder):
    rows=[]
    for (m,s,f,l),frame in data.groupby(['model','scenario','family','level'],sort=True):
        rows.append({'model':m,'scenario':s,'family':f,'level':int(l),**evaluate_metrics(frame)})
    met=pd.DataFrame(rows)
    assert len(met)==len(SELECTED_MODELS)*len(SCENARIOS)
    cat=(met.loc[met.family.ne('clean')].groupby(['model','family'],as_index=False)
        .agg(BA=('BA','mean'),ROC_AUC=('ROC_AUC','mean'),EER=('EER','mean'),
             F1_fake=('F1_fake','mean')))
    summary=[]
    for m,g in met.groupby('model'):
        clean=g.loc[g.family.eq('clean')].iloc[0]
        stress=g.loc[g.family.ne('clean')]
        gc=cat.loc[cat.model.eq(m)]
        assert len(stress)==20 and len(gc)==4
        asri=float(gc.BA.mean())
        summary.append({'model':m,'ASRI_Audio':asri,'clean_BA':float(clean.BA),
                        'drop_clean_to_ASRI':float(clean.BA-asri),
                        'worst_scenario_BA':float(stress.BA.min()),
                        'mean_scenario_AUC':float(stress.ROC_AUC.mean()),
                        'mean_scenario_EER':float(stress.EER.mean()),
                        'clean_ROC_AUC':float(clean.ROC_AUC),'pairs':int(clean.pairs)})
    summ=pd.DataFrame(summary).sort_values('ASRI_Audio',ascending=False)
    base=met.loc[met.scenario.eq('clean'),['model','BA','ROC_AUC','EER']].rename(columns={
        'BA':'BA_clean','ROC_AUC':'AUC_clean','EER':'EER_clean'})
    met=met.merge(base,on='model',validate='many_to_one')
    met['delta_BA_vs_clean']=met.BA-met.BA_clean
    met['delta_AUC_vs_clean']=met.ROC_AUC-met.AUC_clean
    met['delta_EER_vs_clean']=met.EER-met.EER_clean
    met.to_csv(folder/'06_SCENARIO_METRICS.csv',index=False)
    cat.to_csv(folder/'06_CATEGORY_METRICS.csv',index=False)
    summ.to_csv(folder/'06_ASRI_AUDIO_SUMMARY.csv',index=False)
    alert=met.loc[(met.ROC_AUC<=0.1)|(met.ROC_AUC>=0.99)].copy()
    if not alert.empty:
        alert['warning']='Extreme AUC: check preprocessing / orientation / small N; DO NOT automatically invert'
    alert.to_csv(folder/'06_AUC_DIAGNOSTIC_ALERTS.csv',index=False)
    return met,cat,summ,alert

metrics,category,summary,auc_alerts=analyze_predictions(predictions,OUT)
print('ASRI-Audio: середня BA за чотирма групами по п’ять сценаріїв; це адаптація, а не ARI для текстів.')
display(summary)
if len(auc_alerts):
    print('ПОПЕРЕДЖЕННЯ: екстремальний ROC-AUC у',len(auc_alerts),'сценаріях')
    display(auc_alerts[['model','scenario','ROC_AUC','BA','pairs']].head(20))

## 8. Парний bootstrap, порівняння моделей і умовний факторний аналіз

In [ ]:
# Повторні сценарії однієї пари — корельовані. Вибірка bootstrap — PAIR_ID,
# а НЕ тисячі окремих прогнозів. Одна synthetic voice -> CI CONDITIONAL.
B_REPS=min(200,BOOTSTRAP_REPS) if RUN_PHASE=='SMOKE' else BOOTSTRAP_REPS
pair_ids=np.asarray(sorted(predictions.pair_id.astype(str).unique()))
N=len(pair_ids)
models=sorted(SELECTED_MODELS)
arrays={}
for (m,s),g in predictions.groupby(['model','scenario']):
    piv=g.pivot(index='pair_id',columns='label',values='pred_label').reindex(pair_ids)
    assert piv.notna().all().all(),f'Неповні прогнозовані пари {m}/{s}'
    arrays[(m,s)]=piv[[0,1]].to_numpy(dtype=np.int8)

def pair_ba(a,ix):
    v=a[ix]
    return float(.5*(np.mean(v[:,0]==0)+np.mean(v[:,1]==1)))

def asri_on_pairs(model,ix,family=None):
    families=[family] if family else ['noise','mp3','reverb','lowpass']
    cats=[]
    for f in families:
        cats.append(float(np.mean([pair_ba(arrays[(model,s['scenario'])],ix)
                        for s in SCENARIOS if s['family']==f])))
    return float(np.mean(cats))

rng=np.random.default_rng(SEED+77)
boot={m:np.empty(B_REPS,dtype=float) for m in models}
boot_family={(m,f):np.empty(B_REPS,dtype=float) for m in models
             for f in ('noise','mp3','reverb','lowpass')}
for k in range(B_REPS):
    sample=rng.integers(0,N,size=N)
    for m in models:
        vals=[]
        for f in ('noise','mp3','reverb','lowpass'):
            v=asri_on_pairs(m,sample,f)
            boot_family[(m,f)][k]=v
            vals.append(v)
        boot[m][k]=float(np.mean(vals))

cis=[]
for m in models:
    val=summary.set_index('model').loc[m,'ASRI_Audio']
    cis.append({'model':m,'ASRI_Audio':float(val),'ci_lower':float(np.quantile(boot[m],.025)),
                'ci_upper':float(np.quantile(boot[m],.975)),
                'bootstrap_unit':'pair_id','interpretation':'conditional on validation TTS voice'})
ci=pd.DataFrame(cis)
ci.to_csv(OUT/'07_ASRI_PAIR_BOOTSTRAP_CI.csv',index=False)
contrasts=[]
for m1,m2 in itertools.combinations(models,2):
    diff=boot[m1]-boot[m2]
    contrasts.append({'model_A':m1,'model_B':m2,
        'delta_ASRI_A_minus_B':float(summary.set_index('model').loc[m1,'ASRI_Audio']-summary.set_index('model').loc[m2,'ASRI_Audio']),
        'ci_lower':float(np.quantile(diff,.025)),'ci_upper':float(np.quantile(diff,.975)),
        'within_voice_only':True})
pd.DataFrame(contrasts).to_csv(OUT/'07_PAIRWISE_MODEL_CONTRASTS.csv',index=False)

# Діагностика різниць у дії сімейств сценаріїв, а не у процесі навчання.
conditional=[]
try:
    from statsmodels.stats.anova import AnovaRM
    agg=predictions.copy()
    agg['correct']=(agg.label==agg.pred_label).astype(float)
    agg=agg.groupby(['pair_id','model','family'],as_index=False).correct.mean()
    if N>=30 and len(models)>=2:
        res=AnovaRM(agg,depvar='correct',subject='pair_id',within=['model','family']).fit()
        res.anova_table.to_csv(OUT/'07_REPEATED_MEASURES_ANOVA.csv')
        conditional.append('Repeated-measures ANOVA descriptive/conditional on one TTS voice; pair-level data.')
    else:
        conditional.append('SKIPPED repeated-measures ANOVA: fewer than 30 independent pairs or one model.')
except Exception as exc:
    conditional.append('ANOVA unavailable: '+repr(exc))

# Непараметричний Friedman по усередненій коректності кожної пари по категоріях
# (рівень pair, без припущення незалежності окремих 21 сценарію).
fr_rows=[]
correct=predictions.assign(correct=(predictions.label==predictions.pred_label).astype(float))
for m,g in correct.groupby('model'):
    tab=g.groupby(['pair_id','family']).correct.mean().unstack('family')
    if len(tab)<3 or tab.isna().any().any():continue
    cols=['clean','noise','mp3','reverb','lowpass']
    chi,p=stats.friedmanchisquare(*(tab[c].values for c in cols))
    fr_rows.append({'model':m,'N_pairs':len(tab),'Friedman_chi2':float(chi),
                    'p_unadjusted':float(p),'conditional_one_TTS_voice':True})
pd.DataFrame(fr_rows).to_csv(OUT/'07_FAMILY_FRIEDMAN.csv',index=False)
(OUT/'07_STATS_SCOPE.txt').write_text('\n'.join(conditional)+\
    '\nAll uncertainty intervals are conditional on the represented TTS voice(s). '\
    'No cross-generator inference, no human-speaker plus TTS-voice hierarchical CI.\n',encoding='utf-8')
display(ci)
print(*conditional,sep='\n')

## 9. Ранжування моделей, MCDM і чутливість до ваг

In [ ]:
base=metrics.loc[metrics.scenario.eq('clean')].set_index('model').BA
rank_rows=[]
for scenario,g in metrics.groupby('scenario'):
    if scenario=='clean':continue
    current=g.set_index('model').BA
    flips=0;resolutions=0;created=0
    for a,b in itertools.combinations(sorted(base.index),2):
        x=float(base[a]-base[b]);y=float(current[a]-current[b])
        sx=0 if np.isclose(x,0,atol=1e-12) else int(np.sign(x))
        sy=0 if np.isclose(y,0,atol=1e-12) else int(np.sign(y))
        flips+=int(sx*sy<0);resolutions+=int(sx==0 and sy!=0);created+=int(sx!=0 and sy==0)
    rank_rows.append({'scenario':scenario,'pairwise_order_flips':flips,
                      'clean_ties_resolved':resolutions,'new_ties':created})
rev=pd.DataFrame(rank_rows)
rev.to_csv(OUT/'08_RANK_REVERSALS.csv',index=False)

crit=summary.set_index('model')[['ASRI_Audio','clean_BA','worst_scenario_BA','mean_scenario_AUC']].copy()
crit['one_minus_EER']=1-summary.set_index('model').mean_scenario_EER
if crit.isna().any().any():raise RuntimeError('MCDM: NaN критерій')
if len(crit)<2:
    mcdm=pd.DataFrame({'MCDM_consensus':np.ones(len(crit))},index=crit.index)
else:
    span=crit.max()-crit.min()
    z=((crit-crit.min())/span.replace(0,np.nan)).fillna(.5)
    w=np.repeat(1/z.shape[1],z.shape[1])
    rank=crit.rank(ascending=False,method='average')
    srd=1-(rank-1).abs().sum(axis=1)/(len(z.columns)*(len(z)-1))
    dne=1-np.sqrt(np.sum(w*(1-z)**2,axis=1))
    dnm=1-np.sum(w*np.abs(1-z),axis=1)
    wsm=(z*w).sum(axis=1)
    weighted=z*w
    pos=weighted.max(axis=0);neg=weighted.min(axis=0)
    dpos=np.sqrt(((weighted-pos)**2).sum(axis=1))
    dneg=np.sqrt(((weighted-neg)**2).sum(axis=1))
    topsis=(dneg/(dpos+dneg).replace(0,np.nan)).fillna(.5)
    mcdm=pd.DataFrame({'SRD_benefit':srd,'DnE_benefit':dne,'DnM_benefit':dnm,
                       'WSM':wsm,'TOPSIS':topsis})
    assert np.allclose(mcdm.DnM_benefit,mcdm.WSM),'DnM і WSM мали співпасти для L1'
    # У першоджерелі 5 методів, але в цій нормалізації DnM=WSM точно.
    # У консенсусі не подвоюємо один і той же критерій.
    mcdm['MCDM_consensus']=mcdm[['SRD_benefit','DnE_benefit','WSM','TOPSIS']].mean(axis=1)
    # Ранжування при випадкових вагових пріоритетах; ДІАГНОСТИКА, не p-value.
    trials=2000;rng=np.random.default_rng(SEED+121)
    winners={m:0. for m in z.index}
    for weights in rng.dirichlet(np.ones(z.shape[1]),size=trials):
        utility=(z*weights).sum(axis=1)
        tied=utility.loc[np.isclose(utility,utility.max(),atol=1e-12,rtol=0)].index.tolist()
        for m in tied:winners[m]+=1/len(tied)
    sensitivity=pd.DataFrame([{'model':m,'fraction_WSM_first_random_weights':v/trials} for m,v in winners.items()])
    sensitivity.to_csv(OUT/'08_WEIGHT_SENSITIVITY.csv',index=False)
    mcdm=mcdm.sort_values('MCDM_consensus',ascending=False)
mcdm['rank']=mcdm.MCDM_consensus.rank(ascending=False,method='min').astype(int)
mcdm.to_csv(OUT/'08_MCDM_RANKING.csv')
print('Розв’язано ничиїх clean:',int(rev.clean_ties_resolved.sum()),
      '| справжніх парних змін лідерства:',int(rev.pairwise_order_flips.sum()))
display(mcdm)

## 10. Графіки: профілі вразливостей, падіння якості та невизначеність

In [ ]:
# Кожний рисунок має самостійний графік і зберігається у PNG.
heat=metrics.pivot(index='scenario',columns='model',values='BA').reindex(SC.scenario)
fig,ax=plt.subplots(figsize=(8+len(models),10))
im=ax.imshow(heat.to_numpy(float),vmin=0,vmax=1,aspect='auto')
ax.set_yticks(range(len(heat)),heat.index)
ax.set_xticks(range(len(models)),heat.columns,rotation=25,ha='right')
ax.set_title(f'Balanced Accuracy у 21 сценарії (N={N} пар, validation)')
fig.colorbar(im,ax=ax,label='BA [0,1]')
fig.tight_layout();fig.savefig(OUT/'09_HEATMAP_BA.png',dpi=170);plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(10,5))
order=summary.model.tolist()
loc=ci.set_index('model').loc[order]
y=loc.ASRI_Audio.to_numpy(float)
err=np.vstack([y-loc.ci_lower.to_numpy(float),loc.ci_upper.to_numpy(float)-y])
ax.errorbar(range(len(order)),y,yerr=np.maximum(err,0),fmt='o',capsize=5,label='ASRI-Audio; pair bootstrap')
ax.scatter(range(len(order)),summary.set_index('model').loc[order].clean_BA.values,marker='x',label='Clean BA')
ax.set_xticks(range(len(order)),order,rotation=15)
ax.set_ylim(0,1);ax.set_ylabel('Balanced Accuracy / ASRI-Audio [0,1]')
ax.set_title(f'Порівняння clean та сценарної стійкості (N={N} пар; CI within voice)')
ax.grid(axis='y',alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(OUT/'09_ASRI_CLEAN_CI.png',dpi=170);plt.show();plt.close(fig)

for family in ['noise','mp3','reverb','lowpass']:
    df=metrics.loc[metrics.family.eq(family)].copy()
    fig,ax=plt.subplots(figsize=(9,4))
    for m,g in df.groupby('model'):
        ax.plot(g.sort_values('level').level,g.sort_values('level').BA,marker='o',label=m)
    ax.set_xticks(range(1,6));ax.set_ylim(0,1)
    ax.set_xlabel('Рівень спотворення (порядок за зростанням впливу)')
    ax.set_ylabel('Balanced Accuracy [0,1]')
    ax.set_title(f'{family}: вплив інтенсивності на BA (N={N} пар)')
    ax.grid(alpha=.25);ax.legend(fontsize=8);fig.tight_layout()
    fig.savefig(OUT/f'09_CURVE_{family.upper()}_BA.png',dpi=160);plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(10,4))
for m,g in metrics.groupby('model'):
    q=g.sort_values(['family','level'])
    ax.scatter(q.delta_BA_vs_clean,q.delta_AUC_vs_clean,label=m,alpha=.75)
ax.axhline(0,color='grey',lw=.8);ax.axvline(0,color='grey',lw=.8)
ax.set_xlabel('ΔBA до clean');ax.set_ylabel('ΔROC-AUC до clean')
ax.set_title('Зміна порогової якості vs. роздільної здатності при спотвореннях')
ax.grid(alpha=.2);ax.legend();fig.tight_layout()
fig.savefig(OUT/'09_DELTA_BA_VS_AUC.png',dpi=160);plt.show();plt.close(fig)

# Відтворення САМЕ тренувального прикладу, а не перегляд validation/hidden.
if SHOW_TRAIN_AUDIO_EXAMPLE and RUN_PHASE in ('SMOKE','VALIDATION'):
    ex=calibration.iloc[0]
    w=read_wave(ex.path,getattr(ex,'segment_start_sec',0),ex.pair_id)
    scenario=next(s for s in SCENARIOS if s['scenario']=='noise_snr15')
    w2=apply_scenario(w,scenario,str(ex.pair_id),pair_active[str(ex.pair_id)])
    print('Прослуховування ЛИШЕ train; клас',int(ex.label),'пара',str(ex.pair_id))
    display(Audio(w,rate=SR));display(Audio(w2,rate=SR))
    fig,ax=plt.subplots(figsize=(11,3))
    ax.specgram(w2,NFFT=512,Fs=SR,noverlap=384)
    ax.set_xlabel('Час, с');ax.set_ylabel('Частота, Гц')
    ax.set_title('Спектрограма train, noise_snr15')
    fig.tight_layout();fig.savefig(OUT/'09_EXAMPLE_SPECTROGRAM_TRAIN.png',dpi=150);plt.show();plt.close(fig)

## 11. Аудиторський звіт і єдиний ZIP для перевірки

In [ ]:
review={
    'experiment':'scenario-adaptive EVALUATION (not training)','protocol_id':PROTOCOL_ID,
    'publication_reference':'https://doi.org/10.3390/make8060161',
    'run_phase':RUN_PHASE,'full_validation':MAX_PAIRS is None,
    'n_pairs':int(N),'n_models':len(SELECTED_MODELS),
    'n_scenarios':len(SCENARIOS),'n_predictions':len(predictions),
    'training_performed':False,'test_voices':voice_counts.loc[voice_counts.split.eq('validation'),'tts_voice_id'].tolist(),
    'generalization_warning':'Validation contains only one synthetic voice; no cross-generator generalization evidence.',
    'hidden_test_opened':False,
    'extreme_auc_scenarios':len(auc_alerts),
    'stats_scope':'pair bootstrap conditional on validation TTS voice; no population level voice CI'}
write_json_atomic(OUT/'10_REVIEW_SUMMARY.json',review)

# Архів НЕ містить приватного маніфесту, абсолютних шляхів, особистих транскрипцій, WAV, ваг.
# У прогнозах тільки псевдонімні pair_id, модель, scenario, score і ground truth.
ZIP=RESULTS_ROOT/f'SAADE_SCENARIO_EVALUATION_v8_{RUN_TAG}_FOR_REVIEW.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file() and not f.name.startswith('01_PRIVATE_') and f.suffix in ('.csv','.json','.txt','.png'):
            z.write(f,arcname=f.name)
print('Звітний ZIP:',ZIP,'/ MB',round(ZIP.stat().st_size/1e6,2))

## 12. Заморожування та hidden test — **захищена процедура**

**Звичайний режим:** `RUN_PHASE='VALIDATION'`; наведена нижче комірка НЕ відкриває hidden.

**Крок FREEZE:** після наукової перевірки повної validation змініть `RUN_PHASE='FREEZE'`, `FREEZE_CONFIRMATION='I_APPROVE_THE_VALIDATION_PROTOCOL'`, `HIDDEN_CONFIRMATION='NO'`; виконайте весь ноутбук. Заморожується конфігурація, маніфест, калібрування, SHA-256 ваг, сценарії та звітні показники. Зміни після заморожування заборонені.

**Крок HIDDEN_ONCE:** лише після FREEZE змініть `RUN_PHASE='HIDDEN_ONCE'`, `FREEZE_CONFIRMATION='NO'`, `HIDDEN_CONFIRMATION='I_AUTHORIZE_ONE_FINAL_HIDDEN_TEST'`; виконайте весь ноутбук. Одноразовий маркер створюється **до читання прихованих WAV**, під час збою повторний доступ автоматично заборонений (потрібен окремий forensic review), жодні пороги не переналаштовуються.

**Критично:** цей механізм є програмним запобіжником від випадкового повтору, а не криптографічним захистом від ручного втручання у файли. Наявний один hidden TTS-голос не дає підстав заявляти міжгенераторну стійкість.

In [ ]:
FROZEN=BASE/'FROZEN_PROTOCOL_v8.json'
HIDDEN_DIR=BASE/'HIDDEN_ONCE'
HIDDEN_STARTED=BASE/'HIDDEN_ONCE_STARTED.json'
HIDDEN_FINISHED=BASE/'HIDDEN_ONCE_FINISHED.json'

def frozen_payload():
    return {'run_config':RUN_CONFIG,
            'run_config_sha256':sha256_file(CONFIG_PATH),
            'manifest_sha256':sha256_file(MANIFEST_PATH),
            'calibration':cal_meta,'model_provenance':model_provenance,
            'scenario_hash':SCENARIO_HASH,
            'validation_predictions_sha256':sha256_file(PRED_PATH),
            'validation_summary_sha256':sha256_file(OUT/'06_ASRI_AUDIO_SUMMARY.csv')}

if RUN_PHASE=='FREEZE':
    if FREEZE_CONFIRMATION!='I_APPROVE_THE_VALIDATION_PROTOCOL' or HIDDEN_CONFIRMATION!='NO':
        raise RuntimeError('Потрібне окреме явне підтвердження FREEZE; HIDDEN підтвердження вимкнене')
    if MAX_PAIRS is not None or int(N)!=readiness['validation_pairs_total']:
        raise RuntimeError('FREEZE можливий лише після FULL validation без підвибірки')
    data=frozen_payload()
    if not FROZEN.exists():
        with FROZEN.open('x',encoding='utf-8') as f:
            json.dump(data,f,ensure_ascii=False,indent=2)
        print('ПРОТОКОЛ ЗАМОРОЖЕНО:',FROZEN)
    elif json.loads(FROZEN.read_text())!=data:
        raise RuntimeError('Протокол уже заморожений з ІНШИМИ значеннями — НЕ перезаписуйте його')
    else:print('Протокол уже зафіксовано, контрольні суми збігаються.')
elif RUN_PHASE=='HIDDEN_ONCE':
    if HIDDEN_CONFIRMATION!='I_AUTHORIZE_ONE_FINAL_HIDDEN_TEST' or FREEZE_CONFIRMATION!='NO':
        raise RuntimeError('Немає належного ОКРЕМОГО підтвердження HIDDEN_ONCE')
    if not FROZEN.exists():raise RuntimeError('Спочатку створіть FROZEN_PROTOCOL_v8.json у фазі FREEZE')
    frozen=json.loads(FROZEN.read_text(encoding='utf-8'))
    if frozen!=frozen_payload():
        raise RuntimeError('Поточний protocol/checkpoint/threshold/result відрізняється від замороженого')
    if HIDDEN_STARTED.exists() or HIDDEN_FINISHED.exists():
        raise RuntimeError('HIDDEN TEST УЖЕ ВІДКРИВАЛИ. Повторний запуск заборонено, навіть після збою.')
    # Маркер створюємо атомарно ДО першого звернення до прихованих WAV.
    with HIDDEN_STARTED.open('x',encoding='utf-8') as f:
        json.dump({'frozen_sha256':sha256_file(FROZEN),'state':'STARTED (one-time lock)',
                   'warning':'Do not delete this marker; no rerun without independent audit'},f,indent=2)
    print('HIDDEN TEST ВІДКРИТО ОДИН РАЗ. Маркер:',HIDDEN_STARTED)
    h=raw.loc[raw.split.eq('hidden_test')].copy()
    cnt=h.groupby(['pair_id','label']).size().unstack(fill_value=0)
    assert set(cnt.columns)=={0,1} and cnt[[0,1]].eq(1).all().all()
    assert len(h)>0
    HIDDEN_DIR.mkdir(parents=True,exist_ok=True)
    check_file_hashes(h,'hidden').to_csv(HIDDEN_DIR/'HIDDEN_HASH_AUDIT_PRIVATE.csv',index=False)
    for pid,g in h.groupby('pair_id'):
        dur=[sf.info(str(r.path)).duration-float(getattr(r,'segment_start_sec',0) or 0)
             for r in g.itertuples(index=False)]
        pair_active[str(pid)]=int(max(1,min(WINDOW_N,math.floor(SR*min(dur)))))
    hidden_rows=[]
    for name in SELECTED_MODELS:
        detector=load_detector(name)
        try:
            if detector.weights_sha256!=frozen['model_provenance'][name]['weights_sha256']:
                raise RuntimeError('CHECKPOINT ЗМІНЕНО перед hidden')
            threshold=float(frozen['calibration'][name]['threshold'])
            for r in h.itertuples(index=False):
                wave=read_wave(r.path,getattr(r,'segment_start_sec',0),r.pair_id)
                for sc in SCENARIOS:
                    z=transform_cached(wave,sc,r,HIDDEN_DIR/'_mp3_cache')
                    v=detector.score_fake(z)
                    if not np.isfinite(v):raise RuntimeError('NaN hidden prediction')
                    hidden_rows.append({'model':name,'pair_id':str(r.pair_id),
                        'label':int(r.label),'scenario':sc['scenario'],'family':sc['family'],
                        'level':int(sc['level']),'score_fake':float(v),
                        'threshold':threshold,'pred_label':int(v>=threshold)})
        finally:detector.close();del detector;gc.collect()
    hidden=pd.DataFrame(hidden_rows)
    expected_h=len(SELECTED_MODELS)*len(SCENARIOS)*len(h)
    if len(hidden)!=expected_h:raise RuntimeError('Неповний hidden run; marker remains locked')
    hidden.to_csv(HIDDEN_DIR/'HIDDEN_PRIVATE_PREDICTIONS.csv',index=False)
    hmetrics,hcategories,hsummary,halerts=analyze_predictions(hidden,HIDDEN_DIR)
    hreview={'completed':True,'models':SELECTED_MODELS,'pairs':int(h.pair_id.nunique()),
             'prediction_rows':len(hidden),'scenarios':len(SCENARIOS),
             'warning':'single held-out TTS voice; no cross-generator generalization',
             'frozen_sha256':sha256_file(FROZEN)}
    write_json_atomic(HIDDEN_FINISHED,hreview)
    hidden_zip=RESULTS_ROOT/'SAADE_SCENARIO_EVALUATION_v8_HIDDEN_ONCE_FOR_REVIEW.zip'
    with zipfile.ZipFile(hidden_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for f in HIDDEN_DIR.glob('*.csv'):
            if 'PRIVATE' not in f.name:z.write(f,arcname=f.name)
        z.write(HIDDEN_FINISHED,arcname='HIDDEN_ONCE_FINISHED.json')
    print('ОДНОРАЗОВИЙ HIDDEN ЗАВЕРШЕНО. Надіслати:',hidden_zip)
else:
    print('Hidden test ЗАКРИТО. Фаза:',RUN_PHASE)

## 13. Коли готово до статті в **AI (MDPI)**

Збережіть і надішліть `SAADE_SCENARIO_EVALUATION_v8_FULL_VALIDATION_FOR_REVIEW.zip`.

**Мінімум для технічної готовності:** повна validation; три фіксовані pretrained-моделі; 21 сценарій; SHA-256 аудіо та checkpoint; задокументовані пороги лише на train; відсутні leakage; ASRI-Audio, EER, BA, ROC-AUC, F1, CI, MCDM; контроль аномалій ROC-AUC.

**Обмеження для публікаційної новизни:** один TTS-рушій Piper і один голос у validation залишаються суттєвим ризиком. Для сильних міжгенераторних висновків потрібне окреме доповнення аудіоданих із незалежними TTS-рушіями — це **не** робота з навчання моделей.